# ANALYSE-10 : Reflexion sur la these de W. P. Thurston — ce que ce depot en fait

**Serie** : SymbolicAI / Lean — Digestions de resultats profonds
**Auteur source** : William P. Thurston, 1994
**Article de reference** : Thurston, *On proof and progress in mathematics*, Bulletin of the AMS, vol. 30 no. 2 (1994), p. 161-177, arXiv:math/9404236 — bibliotheque partagee, `G:\\Mon Drive\\MyIA\\IA\\Bibliographie IA\\Pedagogie-Proves\\Thurston-1994\\1994 - Thurston - On proof and progress in mathematics (Bull AMS 30, arXiv math 9404236).pdf`
**Contexte** : carnet de distillation reflexif, pli 1 de l'Epic Origami #20404 (issue #20426), grain `DEEP/notebook-python`

## Navigation

| Notebook precedent | Notebook suivant |
|---|---|
| [ANALYSE-09 - Tuilage Aperiodique](ANALYSE-09-Tuilage-Aperiodique.ipynb) | [ANALYSE-11 - Synthese transverse](ANALYSE-11-Synthese-Transverse-Digestions-Python.ipynb) |

***

## Presentation

Ce carnet confronte les theses de W. P. Thurston dans *On proof and progress in mathematics* (1994) a des **donnees mesurees sur ce depot**. Thurston y defend une vision des mathematiques comme **processus humain** et non comme simple accumulation de theoremes : la productivite mathematique, c'est ce que les humains portent, pas ce que la machine verifie.

Les trois theses retenues (sur la demi-douzaine que Thurston developpe) sont celles qui se pretent a une mesure directe par les organes du depot. Une these qui ne peut pas se mesurer est declaree INTRINSIC, avec la raison ecrite dans le carnet.

**Pourquoi ce carnet dans notre serie Lean ?**

- Le pli 1 de l'Origami Thurston demande explicitement un *carnet de distillation reflexif* : non pas un nouveau lake, mais un regard sur l'existant.
- Les organes disponibles (`scripts/lean/count_code_sorry.py`, `scripts/notebook_tools/check_output_collapse.py`, `scripts/notebook_tools/check_source_collapse.py`, `scripts/notebook_tools/check_prose_quantitative_claims.py`) et les traces du prover (`agent_tests/prover/baselines/traces`) donnent **trois** points de mesure distincts : la dette formelle, le processus de production agentique, le ratio formalisation/interpretation.
- Les theses de Thurston sont la matrice pedagogique : comprendre pourquoi nous maintenons un lake (these 1), pourquoi nous coordonnons des agents (these 2), et pourquoi la digestion Python d'un resultat formel peut etre plus volumineuse que le lake lui-meme (these 3).

**Note methodologique** : conformement a la convention de notre serie (cf. ANALYSE-02 Tao, ANALYSE-05 KLS), ce notebook utilise un **kernel Python 3**. Les theoremes Lean ne sont pas executes ; les mesures sont prises firsthand par le notebook. Les sorties reelles (outputs des cellules) sont commitees avec le carnet (C.2 de `notebook-conventions.md`).


## 1. These 1 : la productivite mathematique, c'est ce que les humains portent

**These de Thurston** (paraphrasee) : *le progres en mathematiques se mesure a ce que les humains comprennent et utilisent, pas au nombre de theoremes prouves formellement*. Un lake de Lean sans consommateur de la serie pedagogique est, au sens de Thurston, de la productivite **non-transferee** : la formalisation existe, mais la comprehension n'est pas portee par les autres carnets.

**Mesure** : pour chaque lake, on releve (a) la dette formelle par `count_code_sorry.py`, (b) le nombre de carnets `.ipynb` du depot qui contiennent le chemin du lake (consommateurs), (c) la surface formelle (fichiers `.lean` et LOC). On compare la zone *porteuse* (consommateurs > 0) et la zone *non-portee* (consommateurs = 0).


In [1]:
# Code 1.1 — Productivite formelle vs consommateurs de la serie
#
# On croise count_code_sorry.py (dette formelle), os.walk (consommateurs
# notebooks) et os.walk (surface du lake) pour classer les lakes en zones.

import json
import os
import subprocess

# Le notebook est execute depuis son propre dossier ; on ancre les chemins
# sur la racine du depot, detectee par la presence de .git (3 niveaux au-dessus).
NB_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
# os.getcwd() sera la racine du notebook au moment de l'execution
WORKTREE = os.path.abspath(os.path.join(NB_DIR, '..', '..', '..', '..'))
SORRY_SCRIPT = os.path.join(WORKTREE, 'scripts', 'lean', 'count_code_sorry.py')
NOTEBOOKS_ROOT = os.path.join(WORKTREE, 'MyIA.AI.Notebooks')
SORRY_CACHE = os.path.join(NB_DIR, '_scratch_sorry.json')

# Charger la dette formelle depuis l'organe canonique
if not os.path.exists(SORRY_CACHE):
    r = subprocess.run(
        ['python', SORRY_SCRIPT, '--json'],
        capture_output=True, text=True, encoding='utf-8', errors='replace',
        timeout=120, cwd=WORKTREE
    )
    with open(SORRY_CACHE, 'w', encoding='utf-8') as f:
        f.write(r.stdout)
sorry = json.load(open(SORRY_CACHE, encoding='utf-8'))

# Pre-cache du contenu de tous les ipynb (un seul walk)
ipynb_content = {}
for r, d, f in os.walk(NOTEBOOKS_ROOT):
    for fn in f:
        if fn.endswith('.ipynb'):
            p = os.path.join(r, fn)
            try:
                ipynb_content[p] = open(p, encoding='utf-8', errors='ignore').read()
            except: pass

# Pour chaque lake : consommateurs, fichiers, LOC
lake_info = []
for l in sorry['lakes']:
    lake_path = l['lake']
    consumers = sum(1 for c in ipynb_content.values() if lake_path in c)
    # Walk pour fichiers et LOC (chemin absolu)
    abs_lake = os.path.join(WORKTREE, lake_path)
    file_count = 0
    loc = 0
    if os.path.isdir(abs_lake):
        for r, d, f in os.walk(abs_lake):
            for fn in f:
                if fn.endswith('.lean'):
                    file_count += 1
                    try:
                        loc += sum(1 for _ in open(os.path.join(r, fn), encoding='utf-8', errors='ignore'))
                    except: pass
    lake_info.append({
        'lake': lake_path,
        'sorry': l['distinct_code_sorry'],
        'consumers': consumers,
        'files': file_count,
        'loc': loc,
    })

# Tableau par zone
print('Zone "non-portee" (consommateurs = 0, sorry > 0):')
print(f'  {"lake":<60} {"sorry":>5} {"loc":>6}  files')
for li in sorted(lake_info, key=lambda x: -x['sorry']):
    if li['consumers'] == 0 and li['sorry'] > 0:
        print(f'  {li["lake"]:<60} {li["sorry"]:>5} {li["loc"]:>6}  {li["files"]}')

print()
print('Zone "portee" (consommateurs > 0, sorry > 0):')
print(f'  {"lake":<60} {"sorry":>5} {"consumers":>10} {"loc":>6}')
for li in sorted(lake_info, key=lambda x: -x['sorry']):
    if li['consumers'] > 0 and li['sorry'] > 0:
        print(f'  {li["lake"]:<60} {li["sorry"]:>5} {li["consumers"]:>10} {li["loc"]:>6}')

# Agregat
n_total = len(lake_info)
n_carried = sum(1 for li in lake_info if li['consumers'] > 0)
n_uncarried_sorry = sum(1 for li in lake_info if li['consumers'] == 0 and li['sorry'] > 0)
total_sorry_carried = sum(li['sorry'] for li in lake_info if li['consumers'] > 0)
total_sorry_uncarried = sum(li['sorry'] for li in lake_info if li['consumers'] == 0)
print()
print(f'Total lakes: {n_total}')
print(f'Lakes portes (consommateurs > 0): {n_carried}')
print(f'Lakes non-portes avec sorry: {n_uncarried_sorry}')
print(f'Sorry portes: {total_sorry_carried}')
print(f'Sorry non-portes: {total_sorry_uncarried}')


Zone "non-portee" (consommateurs = 0, sorry > 0):
  lake                                                         sorry    loc  files

Zone "portee" (consommateurs > 0, sorry > 0):
  lake                                                         sorry  consumers    loc
  MyIA.AI.Notebooks/SymbolicAI/Lean/KNOTS/knot_lean                8          2  28693
  MyIA.AI.Notebooks/Probas/DecisionTheory/decision_theory_lean     2          1   3408
  MyIA.AI.Notebooks/GameTheory/game_theory_lean                    1          6  22316
  MyIA.AI.Notebooks/SymbolicAI/Lean/conway_lean                    1          3  41678

Total lakes: 37
Lakes portes (consommateurs > 0): 16
Lakes non-portes avec sorry: 0
Sorry portes: 12
Sorry non-portes: 0


### Lecture du resultat

La zone **non portee** (consommateurs = 0 et `sorry` > 0) est **vide** a cette execution : chaque lake endette a au moins un consommateur. Le cas qui l'illustrerait, `knot_lean` (8 `sorry` reels), en a deux -- `KNOTS-01` (le carnet qui digere knot_lean) et `ANALYSE-11` (la synthese transverse) ; il en compte un troisieme si l'on inclut ce carnet lui-meme, dont la table de sortie publie le chemin de chaque lake endette.

Le 14/10, a la premiere execution, `knot_lean` n'en avait **aucun** et alimentait la zone non portee. Ce n'est pas le lake qui a change, c'est le corpus : `KNOTS-01` a ete resynchronise sur `knot_lean` le 16/10 (#20134) et le cite depuis. Une zone qui se vide chaque fois qu'un carnet se met a citer un lake n'est pas une propriete du depot : c'est une **instantanee**, datee par l'etat du corpus au jour de l'execution.

A l'inverse, `game_theory_lean` et `conway_lean` ont plusieurs consommateurs chacun (cf. Code 1.1) -- la productivite y circule.

**Thurston ici mesure faux** : la these 1, posee comme une these forte, ne dit pas que tout lake non-consomme est un echec. Elle dit que la **mesure de la productivite** ne se reduit pas au sorry=0. Elle reste falsifiable par notre depot -- un lake sans consommateur et sans sorry resterait un lake "isole" au sens de Thurston, et la question de la circulation se poserait -- mais l'instrument qui la teste doit d'abord etre date : un compteur de consommateurs qui depend du corpus, et se compte lui-meme, ne mesure pas une zone stable.


## 2. These 2 : la fiabilite est un processus social

**These de Thurston** (paraphrasee) : *la verification formelle d'un pas de preuve ne suffit pas a la fiabilite d'un enonce mathematique*. La fiabilite emerge du **processus social** qui examine, discute, contredit, revise un resultat. Un verificateur automatique (un noyau de prouveur) execute un controle local ; un processus social distribue la confiance.

**Mesure** : les traces du prover multi-agents (`agent_tests/prover/baselines/traces/*.json`) sont des listes d'evenements `{timestamp, agent, role, content, duration_s}` issues d'une execution reelle. On releve la **diversite des agents** et des **roles** dans une trace representative. La these 2 se mesure par le nombre de participants distincts et la variete des roles, pas par la duree.


In [2]:
# Code 2.1 — Densite sociale des traces du prover
#
# On parcourt les traces, on compte les agents et roles distincts par trace,
# puis on agrege pour caracteriser le "processus social" minimal du prover.

import json
import os
from collections import Counter

# Chemins absolus (cf. Code 1.1)
NB_DIR = os.getcwd()
WORKTREE = os.path.abspath(os.path.join(NB_DIR, '..', '..', '..', '..'))
TRACE_DIR = os.path.join(WORKTREE,
    'MyIA.AI.Notebooks', 'SymbolicAI', 'Lean', 'agent_tests', 'prover', 'baselines', 'traces')

trace_files = sorted([f for f in os.listdir(TRACE_DIR) if f.endswith('.json')])

per_trace = []
all_agents = Counter()
all_roles = Counter()
for f in trace_files:
    data = json.load(open(os.path.join(TRACE_DIR, f), encoding='utf-8'))
    agents = Counter(ev.get('agent', '?') for ev in data)
    roles = Counter(ev.get('role', '?') for ev in data)
    per_trace.append({
        'file': f[:50],
        'events': len(data),
        'agents': len(agents),
        'roles': len(roles),
    })
    all_agents.update(agents)
    all_roles.update(roles)

# Tableau par trace
print(f'Traces: {len(per_trace)}')
print(f'{"events":>6} {"agents":>6} {"roles":>5}  trace')
for t in sorted(per_trace, key=lambda x: -x['events'])[:10]:
    print(f'{t["events"]:>6} {t["agents"]:>6} {t["roles"]:>5}  {t["file"]}')

# Agregat
print()
print(f'Agregat sur les {len(per_trace)} traces:')
print(f'  agents distincts cumules: {len(all_agents)}')
print(f'  roles distincts cumules: {len(all_roles)}')
print(f'  top 5 agents par nombre d\'evenements:')
for ag, c in all_agents.most_common(5):
    print(f'    {ag}: {c}')
print(f'  top 5 roles par nombre d\'evenements:')
for r, c in all_roles.most_common(5):
    print(f'    {r}: {c}')


Traces: 16
events agents roles  trace
    33      6     9  multi_VOTING_MEDIAN_COUNTING_LT_zai.json
    30      4     9  multi_CALIBRATION_CONWAY_LAS_ROUND_TRIP_openrouter
    25      3     9  multi_CALIBRATION_CONWAY_DOOMSDAY_ADD_SEVEN_openro
    25      4     9  multi_CALIBRATION_CONWAY_NIM_SUM_SINGLE_local.json
    22      4     9  multi_CALIBRATION_CONWAY_DOOMSDAY_LEAP_1900_openro
    22      3     7  multi_CALIBRATION_CONWAY_DOOMSDAY_LEAP_2024_openro
    22      3     7  multi_CALIBRATION_CONWAY_LAS_DIGITS_EXAMPLE_openro
    21      5     9  multi_CALIBRATION_CONWAY_ANGEL_CARD_FORMULA_openro
    21      3     7  multi_CALIBRATION_CONWAY_DOOMSDAY_LEAP_2000_openro
    20      3     7  multi_CALIBRATION_CONWAY_DOOMSDAY_CONWAY_DEATH_ope

Agregat sur les 16 traces:
  agents distincts cumules: 7
  roles distincts cumules: 19
  top 5 agents par nombre d'evenements:
    TacticAgent: 181
    SearchAgent: 75
    CoordinatorAgent: 48
    TacticTools: 31
    VerifyExecutor: 3
  top 5 roles pa

### Lecture du resultat

Les traces du prover cumulent plusieurs agents distincts (CoordinatorAgent, SearchAgent, TacticAgent, ProverSetup, VerifyExecutor, TacticTools) et plusieurs roles (`plan`, `compile`, `build_check`, `lost_progress_veto`, `kb_warm_start`, etc.). Une trace individuelle mobilise typiquement plusieurs agents et roles (les comptes exacts sont dans le Code 2.1 ci-dessus). La fiabilite d'un pas de preuve depend de cette **coordination distribuee** : un CoordinatorAgent planifie, un SearchAgent consulte la base de connaissances, un TacticAgent execute, un VerifyExecutor verifie la compilation, un TacticTools assiste le tactic-level.

**Thurston ici gagne** : la these 2 est verifiable par construction. Un noyau de prouveur isole (un seul agent, un seul role) ne reproduit pas ce processus. La fiabilite d'un pas de preuve est, dans notre infrastructure, emergent d'un petit systeme multi-agents. La these 1 et la these 2 se rejoignent : un lake non-consomme (these 1) n'est pas *non-fiable* (le noyau tient), il est non-porte au sens de Thurston. Le prover distribue reste un verificateur local ; la portee sociale reste l'autre agent.


## 3. These 3 : comprendre n'est pas produire une preuve

**These de Thurston** (paraphrasee) : *un mathematicien peut avoir une preuve formelle sans comprendre le resultat, et inversement*. La these 3 est la plus epistemologique des trois. Elle pose que la comprehension est un **acte d'interpretation**, pas une propriete syntaxique d'une chaine de preuves.

**Mesure** : pour un lake formalise, on compare la **surface formelle** (LOC des fichiers `.lean`) a la **surface d'interpretation** (caracteres du carnet Python de digestion, hors enonces). Un ratio eleve signifie que la digestion depasse la formalisation : un humain a passe plus de temps a expliquer qu'a coder. Un ratio bas signifie que la formalisation est nue.

C'est la these la plus delicate a mesurer : la digestion n'est pas quantifiable au mot pres, et le ratio depend du style. Mais elle est **falsifiable** : un lake dont la digestion fait 1% de la surface formelle est un lake "nu", et c'est un point que Thurston aurait signale.


In [3]:
# Code 3.1 -- Ratio formalisation / digestion Python
#
# Pour le lake kls_lean (KLS, formalise localement), on releve LOC du lake
# vs caracteres du carnet de digestion ANALYSE-05. Idem pour knot_lean et
# les 3 carnets KNOTS-01 a 03.
import json
import os

# Chemins absolus (cf. Code 1.1)
NB_DIR = os.getcwd()
WORKTREE = os.path.abspath(os.path.join(NB_DIR, '..', '..', '..', '..'))
NB_ROOT = os.path.join(WORKTREE, 'MyIA.AI.Notebooks')


def measure_notebook(path):
    if not os.path.exists(path):
        return None
    nb = json.load(open(path, encoding='utf-8'))
    md = sum(len(''.join(c['source']) if isinstance(c['source'], list) else c['source'])
             for c in nb['cells'] if c['cell_type'] == 'markdown')
    code = sum(len(''.join(c['source']) if isinstance(c['source'], list) else c['source'])
               for c in nb['cells'] if c['cell_type'] == 'code')
    return {'md': md, 'code': code, 'total': md + code,
            'n_md_cells': sum(1 for c in nb['cells'] if c['cell_type'] == 'markdown'),
            'n_code_cells': sum(1 for c in nb['cells'] if c['cell_type'] == 'code')}


def measure_lake_loc(path):
    if not os.path.isdir(path):
        return 0
    loc = 0
    for r, d, f in os.walk(path):
        for fn in f:
            if fn.endswith('.lean'):
                try:
                    loc += sum(1 for _ in open(os.path.join(r, fn), encoding='utf-8', errors='ignore'))
                except:
                    pass
    return loc


# KLS
kls_lake_path = os.path.join(NB_ROOT, 'SymbolicAI', 'Lean', 'ANALYSE', 'kls_lean')
kls_lake_loc = measure_lake_loc(kls_lake_path)
kls_nb_path = os.path.join(NB_ROOT, 'SymbolicAI', 'Lean', 'ANALYSE', 'ANALYSE-05-KLS-Lean-Python.ipynb')
kls_nb = measure_notebook(kls_nb_path)
print('KLS (theoreme 2026, formalise localement):')
print(f'  lake kls_lean LOC: {kls_lake_loc}')
if kls_nb:
    print(f'  carnet de digestion (KLS): {kls_nb["total"]} chars (markdown={kls_nb["n_md_cells"]}, code={kls_nb["n_code_cells"]})')
    if kls_lake_loc > 0:
        print(f'  ratio digestion / formalisation: {kls_nb["total"] / kls_lake_loc:.1f}x')
print()

# KNOTS
knots_lake_path = os.path.join(NB_ROOT, 'SymbolicAI', 'Lean', 'KNOTS', 'knot_lean')
knots_lake_loc = measure_lake_loc(knots_lake_path)
knots_nbs = []
for f in ['KNOTS-01-Conway-Proofs-Lean-Python.ipynb',
          'KNOTS-02-Invariants-Python.ipynb',
          'KNOTS-03-Companion-Formel-Lean-Python.ipynb']:
    p = os.path.join(NB_ROOT, 'SymbolicAI', 'Lean', 'KNOTS', f)
    nb = measure_notebook(p)
    if nb:
        knots_nbs.append((f, nb))
knots_total_chars = sum(nb['total'] for _, nb in knots_nbs)
print('KNOTS (theorie des noeuds, formalisation lourde):')
print(f'  lake knot_lean LOC: {knots_lake_loc}')
for f, nb in knots_nbs:
    print(f'  {f}: {nb["total"]} chars')
print(f'  total digestion: {knots_total_chars} chars')
if knots_lake_loc > 0:
    print(f'  ratio digestion / formalisation: {knots_total_chars / knots_lake_loc:.1f}x')


KLS (theoreme 2026, formalise localement):
  lake kls_lean LOC: 285
  carnet de digestion (KLS): 35272 chars (markdown=22, code=12)
  ratio digestion / formalisation: 123.8x

KNOTS (theorie des noeuds, formalisation lourde):
  lake knot_lean LOC: 28693
  KNOTS-01-Conway-Proofs-Lean-Python.ipynb: 53338 chars
  KNOTS-02-Invariants-Python.ipynb: 47733 chars
  KNOTS-03-Companion-Formel-Lean-Python.ipynb: 21680 chars
  total digestion: 122751 chars
  ratio digestion / formalisation: 4.3x


### Lecture du resultat

Le lake `kls_lean` (cf. Code 3.1 pour la surface formelle et la longueur de la digestion dans ANALYSE-05) presente un ratio tres eleve (123.8x) -- le mathematicien qui a ecrit la these KLS a depense bien plus d'effort a expliquer la preuve qu'a la formaliser.

Le lake `knot_lean` (cf. Code 3.1) a trois carnets de digestion (KNOTS-01 a 03) pour une formalisation volumineuse : le ratio digestion / formalisation y tombe a 4.3x, contre 123.8x pour `kls_lean` -- un ratio beaucoup plus serre, sans que cela signale un lake nu.

**Thurston ici nuance** : la these 3 ne dit pas que la digestion doit etre plus longue que la formalisation. Elle dit que la comprehension **se voit** dans la digestion, et qu'un lake volumineux avec une digestion modeste n'est pas un lake "nu" -- il est comprehensible via les carnets. KLS est un cas extreme ou la digestion depasse tres largement le message formel, mais ce n'est pas un cas pathologique : c'est un cas ou un mathematicien a reussi a transmettre.

**Limite** : la these 3 est la plus delicate a falsifier. Un lake nu existe (un lake sans digestion), mais on ne peut pas distinguer *a priori* entre un lake peu digere et un resultat intrinsequement clair. La these devient interessante quand on a une digestion longue ET un lake petit (KLS) : la, le ratio parle.


## 4. Exercices

Trois exercices pour approfondir la these de Thurston sur ce depot. Convention C.1 : stubs `pass`/`print`/`return None`, pas de `raise NotImplementedError`.

### 4.1 Exercice 1 -- Classer les lakes du depot en zone portee / non-portee

**Consigne** : etendre le Code 1.1 pour produire un CSV de synthese avec une ligne par lake, colonnes `(lake, sorry, consumers, files, loc, zone)` ou `zone` est `carried` (consommateurs > 0), `uncarried-sorry` (consommateurs = 0 et sorry > 0), ou `uncarried-clean` (consommateurs = 0 et sorry = 0). Discuter : combien de lakes sont dans la zone `uncarried-sorry` ? Est-ce que leur surface (LOC) est comparable a celle des lakes portes ?


In [4]:
# Code 4.1 — Exercice 1 : zones de portage des lakes
#
# L'etudiant doit classer les 36 lakes en 3 zones et compter.
# Stub C.1 : return None avec un print decrivant ce qu'il faudrait faire.

def classify_lakes(sorry_data, ipynb_content):
    # TODO etudiant : classifier chaque lake en 'carried' / 'uncarried-sorry' / 'uncarried-clean'
    # Indice : reprendre la structure de Code 1.1 et ajouter le calcul de zone
    # Indice 2 : utiliser un dict de zones puis Counter pour agreger
    print('Exercice 1 : a implementer par l\'etudiant')
    print('Signature attendue : classify_lakes(sorry_data, ipynb_content) -> dict[zone -> list[lake]]')
    return None  # stub C.1

# Ne pas executer le resultat tant que l'etudiant n'a pas complete.


### 4.2 Exercice 2 — Compter la coordination par trace

**Consigne** : etendre le Code 2.1 pour calculer un **indice de coordination** par trace, defini comme `agents * roles / events`. Discuter : est-ce que les traces avec le plus d'evenements sont aussi celles avec le plus de coordination ? Quel est le role de la these 2 dans cette analyse ?


In [5]:
# Code 4.2 — Exercice 2 : indice de coordination par trace
#
# L'etudiant doit calculer l'indice pour chaque trace, puis le trier.
# Stub C.1 : pass avec un print d'instructions.

import json
import os

def coordination_index(trace_path):
    # TODO etudiant : charger le JSON, compter agents et roles distincts,
    # retourner agents * roles / len(events)
    # Indice : un Counter par attribut, len() sur le Counter pour le distinct
    print(f'Exercice 2 : calculer pour {os.path.basename(trace_path)[:50]}')
    print('Signature : coordination_index(trace_path) -> float')
    return None  # stub C.1

# Verification attendue (non executee par le carnet de l'etudiant) :
# pour multi_CALIBRATION_CONWAY_ANGEL_CARD_FORMULA_openrouter.json,
# 5 agents, 9 roles, 21 events -> 5 * 9 / 21 = 2.14


### 4.3 Exercice 3 — Une these de Thurston qu'on ne peut pas mesurer

**Consigne** : Thurston developpe au moins cinq autres theses que ce carnet n'a pas retenues (par exemple : *mathematical progress is not just "more theorems"*, *the value of a proof is in what it opens up*). En choisir une et expliquer pourquoi elle ne se prete pas a une mesure directe par les organes de ce depot. La these doit etre declaree INTRINSIC, avec la raison ecrite.

**Format attendu** :
- La these (une phrase)
- Pourquoi elle ne se mesure pas ici (au moins 2 raisons)
- Ce qu'il faudrait (autre organe, autre jeu de donnees) pour la rendre mesurable


In [6]:
# Code 4.3 — Exercice 3 : these INTRINSIC
#
# L'etudiant declare verbalement sa these non-mesuree. Stub C.1 : pass.

def intrinsic_thesis():
    # TODO etudiant : retourner un dict avec les cles
    #   - 'these' (la these de Thurston choisie)
    #   - 'motif_INTRINSIC' (pourquoi on ne peut pas la mesurer ici)
    #   - 'orgue_manquant' (ce qu'il faudrait pour la rendre mesurable)
    pass  # stub C.1

# Verification manuelle par le mainteneur : la these doit citer Thurston
# (annee 1994, *On proof and progress in mathematics*) et le motif doit
# nommer explicitement un organe qui n'existe pas.


## 5. Conclusion -- ce que Thurston voit dans ce depot

Les trois theses retenues donnent une lecture a trois niveaux :

- **These 1** (productivite) : la formalisation tient (les `sorry` du depot sont comptes et portes, cf. Code 1.1), mais la portee pedagogique se mesure comme une **instantanee** du corpus : au 19/10, aucun lake endette n'est sans consommateur -- `knot_lean` en etait un jusqu'au 16/10, jusqu'a ce que `KNOTS-01` commence a le citer.

- **These 2** (fiabilite) : la fiabilite n'est pas un noyau de prouveur isole. Le prover multi-agents du depot distribue la confiance entre CoordinatorAgent, SearchAgent, TacticAgent et VerifyExecutor -- c'est un petit systeme social, pas une verification automatique.

- **These 3** (comprendre) : la digestion Python d'un resultat formel peut etre tres largement plus volumineuse que le lake (KLS). Ce n'est pas un defaut ; c'est la these de Thurston mise en pratique : la comprehension se voit.

**Limite de cette lecture** : les trois theses retenues sont celles qui se pretent a une mesure directe par les organes du depot. La moitie des theses de Thurston (le progres comme *shift*, la valeur d'une preuve comme ouverture, le mathematique comme expression culturelle) reste hors mesure, et c'est sain : une these toujours mesurable est une these trop etroite. Le carnet de distillation reflexif se termine la ou la mesure s'arrete, et le pli 2 de l'Origami Thurston prend la suite.

**Voir aussi** : Thurston 1994, *On proof and progress in mathematics* (arXiv:math/9404236, archive partagee `G:\\Mon Drive\\MyIA\\IA\\Bibliographie IA\\Pedagogie-Proves\\Thurston-1994\\`) -- Jaffe & Quinn 1993, *Theoretical mathematics* (arXiv:math/9307227, contre-reponse au plaidoyer de Thurston, archive partagee) -- Thurston 1990, *Mathematical education* (Notices AMS 37, arXiv:math/0503081, archive partagee).

**Issue de rattachement** : #20426, pli 1 de l'Origami #20404. Voir egalement l'EPIC Origami parente et l'instrument `scripts/lean/count_code_sorry.py` pour la these 1, `agent_tests/prover/baselines/traces` pour la these 2, et `MyIA.AI.Notebooks/SymbolicAI/Lean/ANALYSE/` pour la these 3.
